# Generate Embeddings


In [1]:
import json
import pickle
import time
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv(dotenv_path=Path("../.env"))

INPUT_FILE = Path("./chunks.json")
OUTPUT_FILE = Path("./embeddings.pkl")

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1024
BATCH_SIZE = 100

c:\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

child_chunks = [c for c in all_chunks if not c["is_parent"]]
parent_chunks = [c for c in all_chunks if c["is_parent"]]

print(f"Loaded {len(all_chunks)} total chunks")
print(f"  - {len(child_chunks)} child chunks (will be embedded)")
print(f"  - {len(parent_chunks)} parent chunks (stored as context only)")

Loaded 825 total chunks
  - 649 child chunks (will be embedded)
  - 176 parent chunks (stored as context only)


In [8]:
embeddings_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    dimensions=EMBEDDING_DIM,
)

print(f"Initialized embeddings: {EMBEDDING_MODEL}, dim={EMBEDDING_DIM}")

Initialized embeddings: text-embedding-3-small, dim=1024


In [9]:
texts = [c["text"] for c in child_chunks]
all_embeddings = []

total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
start = time.time()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1

    batch_embeddings = embeddings_model.embed_documents(batch)
    all_embeddings.extend(batch_embeddings)

    print(f"Batch {batch_num}/{total_batches} — {len(batch)} texts embedded")

    if batch_num < total_batches:
        time.sleep(0.5)

elapsed = time.time() - start
print(f"\nEmbedded {len(all_embeddings)} chunks in {elapsed:.1f}s")
print(f"Vector dimension: {len(all_embeddings[0])}")

Batch 1/7 — 100 texts embedded
Batch 2/7 — 100 texts embedded
Batch 3/7 — 100 texts embedded
Batch 4/7 — 100 texts embedded
Batch 5/7 — 100 texts embedded
Batch 6/7 — 100 texts embedded
Batch 7/7 — 49 texts embedded

Embedded 649 chunks in 10.0s
Vector dimension: 1024


In [10]:
for chunk, embedding in zip(child_chunks, all_embeddings):
    chunk["embedding"] = embedding

output_data = {
    "child_chunks": child_chunks,
    "parent_chunks": parent_chunks,
}

with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(output_data, f)

file_size_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
print(f"Saved to {OUTPUT_FILE} ({file_size_mb:.1f} MB)")
print(f"  - {len(child_chunks)} child chunks with embeddings")
print(f"  - {len(parent_chunks)} parent chunks (no embeddings)")

Saved to embeddings.pkl (6.8 MB)
  - 649 child chunks with embeddings
  - 176 parent chunks (no embeddings)
